### 1. 要怎麼作實際的應用

In [1]:
import FactorizationMachineModel as fm

VAR_NUM = 8
K = 4
FIELD_DIMS = [2] * 8

model = fm.FactorizationMachineModel(field_dims=FIELD_DIMS, embed_dim=K)

In [2]:
import numpy as np
import torch

class FeaturesEmbedding(torch.nn.Module):

    def __init__(self, field_dims, embed_dim):
        super().__init__()
        self.embedding = torch.nn.Embedding(sum(field_dims), embed_dim)
        self.offsets = np.array((0, *np.cumsum(field_dims)[:-1]), dtype=np.long)
        torch.nn.init.xavier_uniform_(self.embedding.weight.data)

    def forward(self, x):
        x = x + x.new_tensor(self.offsets).unsqueeze(0)
        return self.embedding(x)

class FeaturesLinear(torch.nn.Module):

    def __init__(self, field_dims, output_dim=1):
        super().__init__()
        self.fc = torch.nn.Embedding(sum(field_dims), output_dim)
        self.bias = torch.nn.Parameter(torch.zeros((output_dim,)))
        self.offsets = np.array((0, *np.cumsum(field_dims)[:-1]), dtype=np.long)

    def forward(self, x):
        x = x + x.new_tensor(self.offsets).unsqueeze(0)
        return torch.sum(self.fc(x), dim=1) + self.bias

class FactorizationMachine(torch.nn.Module):

    def __init__(self, reduce_sum=True):
        super().__init__()
        self.reduce_sum = reduce_sum

    def forward(self, x):
        square_of_sum = torch.sum(x, dim=1) ** 2
        sum_of_square = torch.sum(x ** 2, dim=1)
        ix = square_of_sum - sum_of_square
        if self.reduce_sum:
            ix = torch.sum(ix, dim=1, keepdim=True)
        return 0.5 * ix

class FactorizationMachineModel(torch.nn.Module):

    def __init__(self, field_dims, embed_dim):
        super().__init__()
        self.embedding = FeaturesEmbedding(field_dims, embed_dim)
        self.linear = FeaturesLinear(field_dims)
        self.fm = FactorizationMachine(reduce_sum=True)

    def forward(self, x):
        x = self.linear(x) + self.fm(self.embedding(x))
        return x.squeeze(1)

VAR_NUM = 8
K = VAR_NUM
FIELD_DIMS = [2] * VAR_NUM

model = FactorizationMachineModel(field_dims=FIELD_DIMS, embed_dim=K)

### 2. 要怎麼檢查 FM 模型實際內部組成

In [3]:
import FactorizationMachineModel as fm

VAR_NUM = 8
K = 4
FIELD_DIMS = [2] * 8

model = fm.FactorizationMachineModel(field_dims=FIELD_DIMS, embed_dim=K)
print(model)

FactorizationMachineModel(
  (embedding): FeaturesEmbedding(
    (embedding): Embedding(16, 4)
  )
  (linear): FeaturesLinear(
    (fc): Embedding(16, 1)
  )
  (fm): FactorizationMachine()
)


In [4]:
import FactorizationMachineModel as fm

VAR_NUM = 8
K = 4
FIELD_DIMS = [2] * 8

model = fm.FactorizationMachineModel(field_dims=FIELD_DIMS, embed_dim=K)
for name, param in model.named_parameters():
    print(f"{name} : {param.shape}")

embedding.embedding.weight : torch.Size([16, 4])
linear.bias : torch.Size([1])
linear.fc.weight : torch.Size([16, 1])


### 3. 如何將 FM model 作參數的提取

In [5]:
import FactorizationMachineModel as fm

VAR_NUM = 8
K = 4
FIELD_DIMS = [2] * 8

model = fm.FactorizationMachineModel(field_dims=FIELD_DIMS, embed_dim=K)
params = model.state_dict()

print(params['embedding.embedding.weight'], "\n")
print(params['linear.bias'], "\n")
print(params['linear.fc.weight'], "\n")

tensor([[-0.1617, -0.2314, -0.2836, -0.3079],
        [-0.0648,  0.4206,  0.2767, -0.3181],
        [-0.5114, -0.0879, -0.2831, -0.4668],
        [-0.0447,  0.2260, -0.3276, -0.1744],
        [ 0.5326, -0.2987,  0.1566, -0.4203],
        [ 0.2772,  0.5439, -0.3157, -0.2106],
        [ 0.4297,  0.0589, -0.3301,  0.4030],
        [ 0.0489, -0.4310,  0.3475,  0.1667],
        [ 0.3617, -0.2136, -0.2931, -0.3108],
        [ 0.2039, -0.2005, -0.4362, -0.1358],
        [-0.1048, -0.1205,  0.1254, -0.1934],
        [ 0.2760,  0.4661,  0.3729, -0.3933],
        [-0.0036, -0.2872, -0.2444,  0.0402],
        [ 0.3089, -0.4426,  0.4537,  0.5384],
        [ 0.0660,  0.2553,  0.4124, -0.3863],
        [ 0.3067, -0.5238, -0.0780,  0.3572]]) 

tensor([0.]) 

tensor([[ 0.3997],
        [ 0.9650],
        [-0.3876],
        [ 0.8441],
        [-1.5686],
        [-0.1744],
        [-0.4830],
        [-1.1806],
        [ 0.7759],
        [ 0.3785],
        [-1.0626],
        [ 0.3382],
        [ 0.7832],

In [6]:
import FactorizationMachineModel as fm

VAR_NUM = 8
K = 4
FIELD_DIMS = [2] * 8

model = fm.FactorizationMachineModel(field_dims=FIELD_DIMS, embed_dim=K)
params = model.state_dict()

torch.set_printoptions(threshold=float('inf'))

print(params['embedding.embedding.weight'], "\n")
print(params['linear.bias'], "\n")
print(params['linear.fc.weight'], "\n")

tensor([[-0.1769, -0.3327,  0.4494, -0.3264],
        [-0.4036, -0.5052,  0.1814,  0.0844],
        [ 0.1363,  0.1894,  0.1375,  0.4504],
        [ 0.1869,  0.3763, -0.1112,  0.0787],
        [-0.3290, -0.1473, -0.2914, -0.4527],
        [-0.2303,  0.2209, -0.0951, -0.1334],
        [ 0.4742,  0.4872, -0.4678, -0.4934],
        [ 0.0361,  0.2214,  0.1992, -0.0532],
        [-0.2007, -0.1658,  0.2855, -0.4171],
        [ 0.5116, -0.5196, -0.1125,  0.2614],
        [-0.5289, -0.1682, -0.2409, -0.2470],
        [-0.1154,  0.4131, -0.2776,  0.3168],
        [-0.2714, -0.0704,  0.1227, -0.5325],
        [ 0.2948,  0.1183,  0.5279,  0.3708],
        [ 0.2108, -0.1983, -0.3576, -0.0580],
        [ 0.3874, -0.0665, -0.0295, -0.1181]]) 

tensor([0.]) 

tensor([[-0.1805],
        [-0.1945],
        [-0.8963],
        [-0.7447],
        [-0.0115],
        [-0.3020],
        [ 0.7498],
        [ 1.2392],
        [-0.5834],
        [-0.2984],
        [-0.8140],
        [-0.6182],
        [-0.3669],

In [7]:
import FactorizationMachineModel as fm

VAR_NUM = 8
K = 4
FIELD_DIMS = [2] * 8

model = fm.FactorizationMachineModel(field_dims=FIELD_DIMS, embed_dim=K)
params = model.state_dict()
torch.save(model.state_dict(), 'fm_params.pth')

### 4. 要怎麼計算模型內所有參數數量

In [8]:
import FactorizationMachineModel as fm

VAR_NUM = 8
K = 4
FIELD_DIMS = [2] * 8

model = fm.FactorizationMachineModel(field_dims=FIELD_DIMS, embed_dim=K)

for name, param in model.named_parameters():
    print(f"{name} : {param.numel()}")

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total: {trainable_params}")

embedding.embedding.weight : 64
linear.bias : 1
linear.fc.weight : 16
Total: 81


### 5. 如果用原始的耦合方式，變數數量會有差異嗎?

In [9]:
import numpy as np
import torch

class FeaturesLinear_01(torch.nn.Module):
    def __init__(self, field_dims, output_dim=1):
        super().__init__()
        self.fc = torch.nn.Embedding(sum(field_dims), output_dim)
        self.bias = torch.nn.Parameter(torch.zeros((output_dim,)))
        self.offsets = np.array((0, *np.cumsum(field_dims)[:-1]), dtype=np.long)

    def forward(self, x):
        x = x + x.new_tensor(self.offsets).unsqueeze(0)
        return torch.sum(self.fc(x), dim=1) + self.bias


class FactorizationMachine01(torch.nn.Module):
    def __init__(self, field_dims, embed_dim):
        super().__init__()
        self.offsets = np.array((0, *np.cumsum(field_dims)[:-1]), dtype=np.long)
        self.embedding = torch.nn.Embedding(sum(field_dims), embed_dim)
        torch.nn.init.xavier_uniform_(self.embedding.weight.data)

    def forward(self, x):
        x = x + x.new_tensor(self.offsets).unsqueeze(0)
        embed = self.embedding(x)
        num_fields = embed.size(1)
        interaction = []
        for i in range(num_fields):
            for j in range(i + 1, num_fields):
                dot = (embed[:, i, :] * embed[:, j, :]).sum(dim=1)
                interaction.append(dot)

        interaction = torch.stack(interaction, dim=1)
        return interaction.sum(dim=1, keepdim=True)


class FactorizationMachineModel01(torch.nn.Module):
    def __init__(self, field_dims, embed_dim):
        super().__init__()
        self.linear = FeaturesLinear_01(field_dims)
        self.fm = FactorizationMachine01(field_dims, embed_dim)

    def forward(self, x):
        x = self.linear(x) + self.fm(x)
        return x.squeeze(1)

VAR_NUM = 8
K = 4
FIELD_DIMS = [2] * 8

model = FactorizationMachineModel01(field_dims=FIELD_DIMS, embed_dim=K)

for name, param in model.named_parameters():
    print(f"{name} : {param.numel()}")

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total: {trainable_params}")

linear.bias : 1
linear.fc.weight : 16
fm.embedding.weight : 64
Total: 81


In [10]:
import numpy as np
import torch

class FeaturesLinear_02(torch.nn.Module):
    def __init__(self, field_dims, output_dim=1):
        super().__init__()
        self.fc = torch.nn.Embedding(sum(field_dims), output_dim)
        self.bias = torch.nn.Parameter(torch.zeros((output_dim,)))
        self.offsets = np.array((0, *np.cumsum(field_dims)[:-1]), dtype=np.long)

    def forward(self, x):
        x = x + x.new_tensor(self.offsets).unsqueeze(0)
        return torch.sum(self.fc(x), dim=1) + self.bias


class FactorizationMachine02(torch.nn.Module):
    def __init__(self, field_dims, embed_dim):
        super().__init__()
        self.offsets = np.array((0, *np.cumsum(field_dims)[:-1]), dtype=np.long)
        self.embedding = torch.nn.Embedding(sum(field_dims), embed_dim)
        torch.nn.init.xavier_uniform_(self.embedding.weight.data)

    def forward(self, x):
        x = x + x.new_tensor(self.offsets).unsqueeze(0)
        embed = self.embedding(x)
        all_interactions = torch.bmm(embed, embed.transpose(1, 2))
        upper_tri = torch.triu(all_interactions, diagonal=1)
        output = upper_tri.sum(dim=(1, 2), keepdim=True)
        return output.squeeze(2)


class FactorizationMachineModel02(torch.nn.Module):
    def __init__(self, field_dims, embed_dim):
        super().__init__()
        self.linear = FeaturesLinear_02(field_dims)
        self.fm = FactorizationMachine02(field_dims, embed_dim)

    def forward(self, x):
        x = self.linear(x) + self.fm(x)
        return x.squeeze(1)

VAR_NUM = 8
K = 4
FIELD_DIMS = [2] * 8

model = FactorizationMachineModel02(field_dims=FIELD_DIMS, embed_dim=K)

for name, param in model.named_parameters():
    print(f"{name} : {param.numel()}")

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total: {trainable_params}")

linear.bias : 1
linear.fc.weight : 16
fm.embedding.weight : 64
Total: 81


In [11]:
import numpy as np
import torch

class FeaturesEmbedding(torch.nn.Module):

    def __init__(self, field_dims, embed_dim):
        super().__init__()
        self.embedding = torch.nn.Embedding(sum(field_dims), embed_dim)
        self.offsets = np.array((0, *np.cumsum(field_dims)[:-1]), dtype=np.long)
        torch.nn.init.xavier_uniform_(self.embedding.weight.data)

    def forward(self, x):
        x = x + x.new_tensor(self.offsets).unsqueeze(0)
        return self.embedding(x)

class FeaturesLinear(torch.nn.Module):

    def __init__(self, field_dims, output_dim=1):
        super().__init__()
        self.fc = torch.nn.Embedding(sum(field_dims), output_dim)
        self.bias = torch.nn.Parameter(torch.zeros((output_dim,)))
        self.offsets = np.array((0, *np.cumsum(field_dims)[:-1]), dtype=np.long)

    def forward(self, x):
        x = x + x.new_tensor(self.offsets).unsqueeze(0)
        return torch.sum(self.fc(x), dim=1) + self.bias

class FactorizationMachine03(torch.nn.Module):

    def __init__(self, reduce_sum=True):
        super().__init__()
        self.reduce_sum = reduce_sum

    def forward(self, x):
        square_of_sum = torch.sum(x, dim=1) ** 2
        sum_of_square = torch.sum(x ** 2, dim=1)
        ix = square_of_sum - sum_of_square
        if self.reduce_sum:
            ix = torch.sum(ix, dim=1, keepdim=True)
        return 0.5 * ix

class FactorizationMachineModel03(torch.nn.Module):

    def __init__(self, field_dims, embed_dim):
        super().__init__()
        self.embedding = FeaturesEmbedding(field_dims, embed_dim)
        self.linear = FeaturesLinear(field_dims)
        self.fm = FactorizationMachine03(reduce_sum=True)

    def forward(self, x):
        x = self.linear(x) + self.fm(self.embedding(x))
        return x.squeeze(1)

VAR_NUM = 8
K = VAR_NUM
FIELD_DIMS = [2] * VAR_NUM

model = FactorizationMachineModel03(field_dims=FIELD_DIMS, embed_dim=K)

VAR_NUM = 8
K = 4
FIELD_DIMS = [2] * 8

model = FactorizationMachineModel03(field_dims=FIELD_DIMS, embed_dim=K)

for name, param in model.named_parameters():
    print(f"{name} : {param.numel()}")

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total: {trainable_params}")

embedding.embedding.weight : 64
linear.bias : 1
linear.fc.weight : 16
Total: 81


### 6. 延續上題，做運算時長得比較

In [12]:
import torch
import torch.utils.benchmark as benchmark

VAR_NUM    = 100
K          = 50
FIELD_DIMS = [2] * VAR_NUM
BATCH_SIZE = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Use: {device}\n")

model_loop    = FactorizationMachineModel01(FIELD_DIMS, K).to(device).eval()
model_triu    = FactorizationMachineModel02(FIELD_DIMS, K).to(device).eval()
model_formula = FactorizationMachineModel03(FIELD_DIMS, K).to(device).eval()

x = torch.randint(0, 2, (BATCH_SIZE, VAR_NUM), device=device)

timer_loop = benchmark.Timer(
    stmt='model_loop(x)',
    globals={'model_loop': model_loop, 'x': x},
    label='Factorization Machine', 
    sub_label='1. Loop',
    description='Forward Pass'
)

timer_triu = benchmark.Timer(
    stmt='model_triu(x)',
    globals={'model_triu': model_triu, 'x': x},
    label='Factorization Machine', 
    sub_label='2. Triu',
    description='Forward Pass'
)

timer_formula = benchmark.Timer(
    stmt='model_formula(x)',
    globals={'model_formula': model_formula, 'x': x},
    label='Factorization Machine', 
    sub_label='3. Formula',
    description='Forward Pass'
)

results = [
    timer_loop.timeit(number=100),
    timer_triu.timeit(number=100),
    timer_formula.timeit(number=100),
]

benchmark.Compare(results).print()

Use: cuda

[---- Factorization Machine ----]
                  |  Forward Pass
1 threads: ----------------------
      1. Loop     |    212301.0  
      2. Triu     |      1066.8  
      3. Formula  |       644.5  

Times are in microseconds (us).

